In [12]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import pymc as pm
import arviz as az
import pandas as pd
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional
import random
import itertools
import warnings
import math

# Notebook config
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import simulation_engine.enums as enums
import simulation_engine.models as models
import simulation_engine.initialization as initialization
import simulation_engine.deck as deck
import simulation_engine.mechanics as mechanics
import simulation_engine.agents as agents
import simulation_engine.engine as engine

# Board Game Simulation

## Volcano Rush

### Abstract

*(To be filled at the end of the project)*

- Summary of simulation approach and scope
- Key findings: win rates, character balance, resource dynamics
- Main conclusions on game design

### Introduction

Game balance is considered critical to the success of any game, yet there is no universal consensus on what it actually means [1]. It can be broadly interpreted as the process of tuning a game's rules, difficulty, play time, resources, and role abilities so that no single strategy, character, or configuration dominates the experience. For multiplayer games in particular, balance has several dimensions: ensuring no starting position is inherently advantageous, preventing any strategy from being strictly dominant, and calibrating the cost-to-benefit ratio across different game objects [1].

As a semi-cooperative board game, *Volcano Rush* faces a specific balancing challenge: the group must share a win condition (escaping the volcano), while players also compete for individual scores. A well-balanced game therefore needs a group win rate that feels achievable but not trivial, and individual scores that are not dominated by a single character role. Compounding this, the game supports 4 to 8 players, meaning balance must hold across a wide range of player configurations.

As with most game design problems, there is no deterministic recipe for achieving balance. *Volcano Rush* is both strategic and social, and raw numbers can sometimes mislead. Simulation is not a substitute for playtesting, but it provides a principled starting point for identifying where imbalances may arise before bringing the game to the table.

### Related Work

#### Monte Carlo simulation

Monte Carlo simulation estimates probabilities by running many random trials instead of solving mathematical formulas, which can be extremely difficult for complex games. Running $n$ simulated games under a given strategy $M$ yields an empirical win probability:

We define an indicator variable for each simulation:

- $X_i = 1$ if simulation $i$ results in a win  
- $X_i = 0$ otherwise  

The empirical win probability after $n$ simulations is:

$$
\hat{p}_n(M) = \frac{1}{n} \sum_{i=1}^{n} X_i
$$

As the number of simulations increases, this estimate converges to the true win probability:

$$
p(M) = \lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^{n} X_i
$$

This follows from the **Law of Large Numbers**, which states that the sample average converges to the expected value (true probability) as the number of trials grows. This makes Monte Carlo methods a good fit for games with many possible situations, where computing the exact answer is not practical.

#### Monte Carlo Tree Search (MCTS)

Monte Carlo methods are also widely used in game AI through Monte Carlo Tree Search (MCTS) [2], which builds a tree of possible future game states and tests them with simulations. 
At each step, the algorithm faces a trade-off between two ideas:
- *exploitation* - choosing moves that have already shown good results  
- *exploration* - trying moves that have not been tested much but might be better  

This balance is handled by the **Upper Confidence Bound for Trees** (UCT) formula:

$$ \mathrm{UCT} = \frac{W_i}{N_i} + c \sqrt{\frac{\ln N_p}{N_i}} $$

The formula combines two parts:
- $\frac{W_i}{N_i}$ - the **exploitation term**, which represents how successful a move has been so far (its average win rate)
- $c \sqrt{\frac{\ln N_p}{N_i}}$ - the **exploration term**, which gives a higher value to moves that have been tried fewer times, encouraging the algorithm to explore them
- $c$ - a constant that controls the balance between exploration and exploitation (higher values lead to more exploration)

In practice, MCTS does not explore all possible game paths. Instead, it gradually builds the search tree through repeated simulations. At the beginning, it gathers initial data by trying different possible moves. As more simulations are performed, the algorithm starts to focus more on the moves that show better results, exploring them in greater depth. However, less-explored moves are still occasionally tested due to the exploration term in the UCT formula, ensuring that potentially good alternatives are not missed.

In this work, however, we use simpler repeated simulations with rule-based agents to study game balance rather than to improve decision-making [2].

#### Agent-Based Modeling (ABM)

In Agent-Based Modeling (ABM), each agent follows a set of simple rules and interacts with other agents and the world around it [3]. Over time, these small interactions can produce complex group behaviors that would be hard to predict just by looking at the rules alone.

Compared to pure Monte Carlo simulation, ABM allows for more realistic behavior. Monte Carlo is fully random, but in ABM, agents can follow rules that reflect real human decisions. This approach has been used in research: Deadman, Schlager, and Gimblett (2000) interviewed participants after a game experiment and used their answers to design the decision rules for ABM agents [5]. For *Volcano Rush*, we could do the same - interview players after a playtest session and use what we learn to make the agents behave more like real people [4].

Traditional game theory assumes that players always make the best possible decision to get the best outcome for themselves. But in real life, people are often influenced by emotions, habits, and not having all the information [3]. This means game theory models do not always reflect how real people actually play.

### Data

This study uses no external dataset. All data is generated by the simulation itself - each run of the simulation produces one game record. The input to the simulation is a fixed set of game parameters taken directly from the game rules.

#### Game Parameters

The resource deck contains **60 cards** split equally between three types: Wood, Stone, and Rope (20 each). The deck is reshuffled and reused when it runs out. The camp has two **shared tools** - Knife and Vessel - which can become damaged and must be repaired before use.

There are **6 character roles**: Builder, Fire Starter, Craftsman, Cook, Gatherer, and Sailor. Each has a unique ability that affects resource requirements, point rewards, or complication handling. For games of 4 or 5 players, the Craftsman role is always assigned and the remaining roles are drawn randomly. For 6 players, all six roles are used once. For 7–8 players, one or two roles are repeated.

The **mission pool** consists of 8 standard missions and up to 5 boat missions. At any time, exactly 3 missions are active and new missions are drawn when one is completed or discarded. The number of boat missions in play - and the number of boat parts needed to win - depends on player count:

| Players | Boat Parts Required | Active Boat Missions |
|---|---|---|
| 4–5 | 2 | 2 randomly selected |
| 6 | 4 | All 4 base missions |
| 7–8 | 5 | All 4 base missions + Fit the Rudder |

The **Complication deck** contains 11 cards (including 2 neutral cards with no effect) and is reshuffled when exhausted. The **Volcano deck** contains 11 penalty cards with the Eruption card fixed at the bottom - when it is drawn, the game ends in a loss.
Card draws from the resource, complication, and volcano decks are modeled as stochastic processes with uniform random sampling.

#### Simulation Assumptions

Where the game rules leave room for interpretation, the following assumptions are applied:

| Rule Area | Assumption |
|---|---|
| Exhaustion | Applies to all participants after resolution, regardless of success or failure |
| Craftsman repair | Starts round N, tool unavailable round N+1, available again round N+2 |
| Sailor - lesser evil | The less severe Complication card is chosen using a severity scoring function |
| Participant count | Treated as an exact requirement, no additional participants allowed |
| Damaged tool | A mission requiring a damaged tool automatically fails |
| Night Anxiety | Requires 1 non-participant helper; if none available, mission fails |
| Gather - default | Any player not participating automatically gathers |
| Gather - non-Gatherer | Drawing 1 resource via Gather does not cause Exhaustion |
| Gather - Gatherer role | Using the boosted draw (2 resources) causes Exhaustion |

#### Output Variables

Each simulation run records: game outcome (win/loss), number of rounds played, individual score per player, win rate per role (role dominance), number of boat parts completed, and number of volcano cards remaining when the game ends.

### Methods

The simulation runs each game as a sequence of rounds following the 6-step loop described in `docs/game_rules.md`. Thousands of games are run per configuration, and outcomes are recorded to estimate win rates and individual scores empirically.

#### Agent Strategy

Each non-exhausted player agent follows a set of rule-based heuristics that approximate plausible human behavior. The rules apply in priority order each round:

**Mission voting (before the round starts)**
- Each agent votes for the mission where their character role has a direct ability bonus (e.g. Builder votes for Wood-heavy missions, Cook votes for Vessel missions, Fire Starter votes for "Light a Fire" or "Torch"). The mission with the most votes is selected.
- If no agent has a preference, a mission is chosen at random from the 3 active ones.

**Action decision (participate or gather)**
1. **Participate** - if the agent is not Exhausted and has enough resources to contribute the required amount and still keep at least 1 card in hand.
2. **Gather** - if the agent cannot or chooses not to participate.
   - **Gatherer role**: uses the boosted draw (2 resources) when not Exhausted and resources are below a threshold; otherwise gathers normally.

**Craftsman repair rule**
- If the Craftsman is not Exhausted and at least 1 tool is damaged, they skip the mission and start a repair instead.

**Boat mission priority**
- When the volcano deck has 4 or fewer cards remaining, all agents give maximum priority to boat missions in their voting.

**Low-pressure rules (only apply when volcano deck has more than 4 cards remaining)**
- *Resource conservation*: agents with only 1 card in hand always gather instead of participating.
- *Exhaustion spreading*: if more than half the team is Exhausted, non-Exhausted agents prefer to gather to rebuild resources for the next round.

#### Simulation Scenarios

Games are run across all player counts (4-8). Character assignment follows the game rules: Craftsman is always assigned in 4-5 player games; all 6 roles are used exactly once in 6-player games; repeated roles fill the extra slots in 7-8 player games. Each scenario runs N = 1,000-10,000 games. Sensitivity analysis varies the participation threshold and boat mission priority trigger.

#### Statistical Analysis

Win rates per scenario are estimated using a Bayesian Beta-Binomial model (PyMC). Posterior distributions are compared across player counts and character compositions using ArviZ credible intervals.

#### Configure Simulation Scenarios

- Scenarios by player count: 4, 5, 6, 7, 8
- Character composition: random assignment vs. fixed "strong" team vs. fixed "weak" team
- Strategy variants: aggressive (always participate) vs. conservative (gather-heavy)
- Number of runs per scenario: N = 1,000–10,000 (tune based on variance)

In [ ]:
results = {}
for pc in [4, 5, 6, 7, 8]:
    results[pc] = engine.run_scenario(player_count = pc, n_games = 1000, base_seed = 42)

#### Exploratory Simulation Runs

- Single-game trace to verify logic (print round-by-round state)
- Check distribution of game lengths (rounds to win or lose)
- Sanity checks: resource deck exhaustion rate, average volcano cards drawn per game

In [ ]:
import random
import matplotlib.pyplot as plt

# ── Single-game verbose trace ──────────────────────────────────────────────

random.seed(0)
state = initialization.init_game(player_count = 5)
print(f"Characters: {[p.character.value for p in state.players]}")
print(f"Boat parts required: {state.boat_parts_required}")
print(f"Active missions: {[m.value for m in state.active_missions]}")
print()

game_over = False
won       = False
while not game_over:
    prev_round     = state.round
    prev_volcano   = len(state.volcano_deck)
    prev_parts     = len(state.boat_parts_built)
    prev_missions  = list(state.active_missions)
    prev_scores    = [p.score for p in state.players]

    game_over, won = engine.run_round(state)

    mission_delta = [m for m in prev_missions if m not in state.active_missions]
    parts_delta   = len(state.boat_parts_built) - prev_parts
    volcano_used  = prev_volcano - len(state.volcano_deck)
    scores        = [p.score for p in state.players]
    score_gains   = [scores[i] - prev_scores[i] for i in range(len(scores))]

    completed = mission_delta[0].value if mission_delta else "—"
    print(
        f"Round {state.round:>2} | mission: {completed:<28} "
        f"| volcano left: {len(state.volcano_deck):>2} (used {volcano_used}) "
        f"| boat: {len(state.boat_parts_built)}/{state.boat_parts_required} "
        f"| scores: {scores} (+{score_gains})"
    )

print()
print(f"Outcome: {'WIN' if won else 'LOSS'} after {state.round} rounds")
print(f"Final scores: { {p.character.value: p.score for p in state.players} }")

# ── Win rates per player count ─────────────────────────────────────────────

player_counts = sorted(results.keys())
win_rates     = [
    sum(1 for r in results[pc] if r.outcome == "win") / len(results[pc]) * 100
    for pc in player_counts
]

fig, axes = plt.subplots(1, 3, figsize = (15, 5))

axes[0].bar([str(pc) for pc in player_counts], win_rates, color = "steelblue")
axes[0].set_xlabel("Player count")
axes[0].set_ylabel("Win rate (%)")
axes[0].set_title("Win rate by player count")
axes[0].set_ylim(0, 100)
for i, v in enumerate(win_rates):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha = "center", fontsize = 9)

# ── Game length distribution (wins only) ─────────────────────────────────

all_lengths = [r.rounds_played for pc in player_counts for r in results[pc] if r.outcome == "win"]
axes[1].hist(all_lengths, bins = 20, color = "seagreen", edgecolor = "white")
axes[1].set_xlabel("Rounds played")
axes[1].set_ylabel("Games")
axes[1].set_title("Game length distribution (wins)")

# ── Avg volcano cards remaining at game end ───────────────────────────────

avg_volcano = [
    sum(r.volcano_cards_remaining for r in results[pc]) / len(results[pc])
    for pc in player_counts
]
axes[2].bar([str(pc) for pc in player_counts], avg_volcano, color = "tomato")
axes[2].set_xlabel("Player count")
axes[2].set_ylabel("Cards remaining (avg)")
axes[2].set_title("Avg volcano cards remaining at end")

plt.tight_layout()
plt.show()

# ── Sanity checks ─────────────────────────────────────────────────────────

print("\nSanity checks")
print("─" * 50)
for pc in player_counts:
    games      = results[pc]
    wins       = [r for r in games if r.outcome == "win"]
    losses     = [r for r in games if r.outcome == "loss"]
    avg_rounds = sum(r.rounds_played for r in games) / len(games)
    avg_vc     = sum(r.volcano_cards_remaining for r in games) / len(games)
    avg_parts  = sum(r.boat_parts_built for r in games) / len(games)
    print(
        f"  {pc}p | wins: {len(wins):>4}/{len(games)} ({len(wins)/len(games)*100:5.1f}%) "
        f"| avg rounds: {avg_rounds:5.1f} "
        f"| avg volcano left: {avg_vc:4.1f} "
        f"| avg boat parts: {avg_parts:.2f}/{games[0].boat_parts_required}"
    )

#### Visualize Outcomes

- Win rate by player count (bar chart with confidence intervals)
- Distribution of game length (histogram)
- Character score distributions (violin or box plot per character)
- Volcano deck depth at game end (how close to eruption)
- Resource economy: average cards in hand per round
- Mission completion heatmap (which missions complete most often)

In [ ]:
# Plots


#### Statistical Analysis

- Bayesian estimation of win rate per scenario using PyMC Beta-Binomial model
- Posterior comparison: does player count 6 win significantly more often than 4?
- Character dominance: Bayesian comparison of individual scores per character
- ArviZ for posterior visualization and credible intervals

In [ ]:
import pymc as pm
import arviz as az

# PyMC models + ArviZ plots

### Results

- Win rates across player counts - is it easier with more players?
- Which characters emerge as score leaders? Is there a dominant strategy?
- Resource bottlenecks: which resource type runs out most often?
- How often does the volcano erupt vs. the group escaping?
- Boat mission difficulty ranking (which cause the most failures?)
- Complication card impact - which ones most often flip a win to a loss?

### Discussion / Further Work

- Is the game balanced as designed, or does it favor a specific player count?
- Suggestions for rule tweaks to improve balance (adjust required boat parts, resource deck size)
- Limitations: heuristic strategies don't capture human creativity or negotiation
- Next steps:
  - Replace rule-based agents with learned strategies (RL or MCTS)
  - Explore asymmetric information (players don't see each other's hands)
  - Sensitivity to rule variants (e.g., exhaustion only on success)
  - Compare simulated win rates against real playtest results

### References / Bibliography

[1] [Schell, J. (2009). Level 16: Game Balance. Game Design Concepts.](https://gamedesignconcepts.wordpress.com/2009/08/20/level-16-game-balance/)

[2] [Chaslot, G., Bakkes, S., Szita, I., & Spronck, P. (2008). Monte-Carlo Tree Search: A New Framework for Game AI. AIIDE.](https://sander.landofsand.com/publications/AIIDE08_Chaslot.pdf)

[3] [StudyGuides.com. (2026). Simulations in Game Theory.](https://studyguides.com/study-methods/overview/clzggcm6j0j8t8cfe9zugo2t8)

[4] [Dubois, E., Barreteau, O., & Souchère, V. (2013). An Agent-Based Model to Explore Game Setting Effects on Attitude Change During a Role Playing Game Session. Journal of Artificial Societies and Social Simulation, 16(1), 2.](https://www.jasss.org/16/1/2.html)

[5] [Yiannakoulias, N., Grignon, M., & Marshall, T. (2024). Parameterizing agent-based models using an online game. Computers, Environment and Urban Systems, 112, 102142.](https://www.sciencedirect.com/science/article/pii/S0198971524000711)

### Appendix

- **Appendix A:** General game rules (overview, resources, mechanics, scoring) - `docs/game_rules.md`
- **Appendix B:** Mission table - `docs/missions.md`
- **Appendix C:** Complication card descriptions - `docs/complication_cards.md`
- **Appendix D:** Volcano card descriptions - `docs/volcano_cards.md`
- **Appendix E:** Character ability reference - `docs/characters.md`
- **Appendix F:** Raw simulation output tables *(to be filled after simulation runs)*